In [8]:
from openai import OpenAI
from typing import Optional, List, Mapping, Any, Sequence, Dict
from llama_index.core.bridge.pydantic import Field, PrivateAttr
from llama_index.core.constants import DEFAULT_CONTEXT_WINDOW, DEFAULT_NUM_OUTPUTS
from llama_index.core.llms import (
    CustomLLM,
    CompletionResponse,
    CompletionResponseGen,
    LLMMetadata,
    ChatMessage,
    ChatResponse,
    MessageRole
)
from llama_index.core.llms.callbacks import llm_completion_callback, llm_chat_callback
from llama_index.core.embeddings import BaseEmbedding

In [9]:
def to_message_dicts(messages: Sequence[ChatMessage])->List:
    return [
        {"role": message.role.value, "content": message.content,} 
                for message in messages 
    ]

def get_additional_kwargs(response) -> Dict:
    return {
        "token_counts":response.usage.total_tokens,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
    }

class ChatGLM(CustomLLM):
    num_output: int = DEFAULT_NUM_OUTPUTS
    context_window: int = Field(default=DEFAULT_CONTEXT_WINDOW,description="The maximum number of context tokens for the model.",gt=0,)
    model: str = Field(default=None, description="The llm model to use")
    api_key: str = Field(default=None, description="The LLM API key.")
    base_url: str = Field(default=None, description="The LLM Base Url")
    top_p: float = Field(default=0, description="The LLM Base Url"),
    temperature:float = Field(default=0, description="The LLM Base Url"),
    reuse_client: bool = Field(default=True, description=(
            "Reuse the client between requests. When doing anything with large "
            "volumes of async API calls, setting this to false can improve stability."
        ),
    )

    _client: Optional[Any] = PrivateAttr()
    def __init__(
        self,
        model: str = None,
        reuse_client: bool = True,
        api_key: Optional[str] = None,
        base_url:str= None,
        top_p:float = 0.6,
        temperature:float = 0.6,
        **kwargs: Any,
    )-> None:
        super().__init__(
            model=model,
            api_key=api_key,
            reuse_client=reuse_client,
            base_url=base_url,
            top_p = 0.6,
            temperature=0.6,
            **kwargs,
        )
        self._client = None
        self.top_p=top_p
        self.temperature=temperature

    def _get_client(self) -> OpenAI:
        if not self.reuse_client :
            return OpenAI(api_key=self.api_key,base_url=self.base_url)

        if self._client is None:
            self._client = OpenAI(api_key=self.api_key,base_url=self.base_url)
        return self._client

    @classmethod
    def class_name(cls) -> str:
        return "chatglm_llm"

    @property
    def metadata(self) -> LLMMetadata:
        """Get LLM metadata."""
        return LLMMetadata(
            context_window=self.context_window,
            num_output=self.num_output,
            model_name=self.model,
        )

    def _chat(self, messages:List, stream=False) -> Any:
        #print("--------------------------------------------")
        #import traceback
        #s=traceback.extract_stack()
        # print("%s %s invoke _chat" % (s[-2],s[-2][2]))
        # print(messages)
        # print("--------------------------------------------")
        response = self._get_client().chat.completions.create(
            model=self.model,  # 填写需要调用的模型名称
            messages=messages,
            top_p=self.top_p, 
            temperature=self.temperature  # 0-1  
        )
        # print(f"_chat, response: {response}")
        return response

    #@llm_chat_callback()
    def chat(self, messages: Sequence[ChatMessage], **kwargs: Any) -> ChatResponse:
        message_dicts: List = to_message_dicts(messages)
        response = self._chat(message_dicts, stream=False)
        # print(f"chat: {response} ")
        rsp = ChatResponse(
            message=ChatMessage(content=response.choices[0].message.content, role=MessageRole(response.choices[0].message.role),
                additional_kwargs= {}),
            raw=response, additional_kwargs= get_additional_kwargs(response),
        )
        print(f"chat: {rsp} ")

        return rsp

    #@llm_chat_callback()
    def stream_chat(self, messages: Sequence[ChatMessage], **kwargs: Any) -> CompletionResponseGen:
        response_txt = ""
        message_dicts: List = to_message_dicts(messages)
        response = self._chat(message_dicts, stream=True)
        # print(f"stream_chat: {response} ")
        for chunk in response:
            # chunk.choices[0].delta # content='```' role='assistant' tool_calls=None
            token = chunk.choices[0].delta.content
            response_txt += token
            yield ChatResponse(message=ChatMessage(content=response_txt,role=MessageRole(message.get("role")),
                                additional_kwargs={},), delta=token, raw=chunk,)

    @llm_completion_callback()
    def complete(self, prompt: str, **kwargs: Any) -> CompletionResponse:
        messages = [{"role": "user", "content": prompt}]
        # print(f"complete: messages {messages} ")
        try:
            response = self._chat(messages, stream=False)

            rsp = CompletionResponse(text=str(response.choices[0].message.content), 
                                     raw=response, 
                                     additional_kwargs=get_additional_kwargs(response),)
            # print(f"complete: {rsp} ")
        except Exception as e:
            print(f"complete: exception {e}")

        return rsp

    @llm_completion_callback()
    def stream_complete(self, prompt: str, **kwargs: Any) -> CompletionResponseGen:
        response_txt = ""
        messages = [{"role": "user", "content": prompt}]
        response = self._chat(messages, stream=True)
        # print(f"stream_complete: {response} ")
        for chunk in response:
            # chunk.choices[0].delta # content='```' role='assistant' tool_calls=None
            token = chunk.choices[0].delta.content
            response_txt += token
            yield CompletionResponse(text=response_txt, delta=token)

In [10]:
API_KEY="d831f56604634f339e526a8b02bd603a.PVwAtiMLDNEFr6ot"
BASE_URL="https://open.bigmodel.cn/api/paas/v4"
DEFAULT_MODEL="GLM-4"
test_llm = ChatGLM(model=DEFAULT_MODEL, reuse_client=True, api_key=API_KEY,base_url=BASE_URL)
test_messages = [{"role": "user", "content": "讲个简短的笑话"}]
test_llm._chat(test_messages)

ChatCompletion(id='2026070113125647cad3364f364a94', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='好的，给你一个：\n\n有一天，包子和打架，包子被包子打了一顿。\n包子觉得很委屈，就回家找它哥哥花诉苦。\n花觉得包子太可怜了，就带着它兄弟们一起去找包子报仇。\n你知道为什么吗？\n因为它身上有“纹身”。\n\n希望这个笑话能给你带来一点欢乐！', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1782882777, model='glm-4', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=69, prompt_tokens=14, total_tokens=83, completion_tokens_details=None, prompt_tokens_details=None), request_id='2026070113125647cad3364f364a94')

In [11]:
class ChatGLMEmbeddings(BaseEmbedding):
    model: str = Field(default='text-embedding-3', description="The ChatGLM model to use.")
    api_key: str = Field(default=None, description="The ChatGLM API key.")
    base_url: str = Field(default=None, description="The LLM Base Url")
    reuse_client: bool = Field(default=True, description=(
            "Reuse the client between requests. When doing anything with large "
            "volumes of async API calls, setting this to false can improve stability."
        ),
    )

    _client: Optional[Any] = PrivateAttr()

    def __init__(
        self,
        model: str = 'text-embedding-3',
        reuse_client: bool = True,
        api_key: Optional[str] = None,
        base_url: Optional[str] = None,
        **kwargs: Any,
    ) -> None:
        super().__init__(
            model=model,
            api_key=api_key,
            base_url=base_url,
            reuse_client=reuse_client,** kwargs,
        )
        self._client = None

    def _get_client(self) -> OpenAI:
        if not self.reuse_client:
            return OpenAI(api_key=self.api_key, base_url=self.base_url)
        if self._client is None:
            self._client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        return self._client

    @classmethod
    def class_name(cls) -> str:
        return "ChatGLMEmbedding"

    def _get_query_embedding(self, query: str) -> List[float]:
        return self.get_general_text_embedding(query)

    async def _aget_query_embedding(self, query: str) -> List[float]:
        return self.get_general_text_embedding(query)

    def _get_text_embedding(self, text: str) -> List[float]:
        return self.get_general_text_embedding(text)

    async def _aget_text_embedding(self, text: str) -> List[float]:
        return self.get_general_text_embedding(text)

    def _get_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        embeddings_list: List[List[float]] = []
        for text in texts:
            embeddings = self.get_general_text_embedding(text)
            embeddings_list.append(embeddings)
        return embeddings_list

    async def _aget_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        return self._get_text_embeddings(texts)

    def get_general_text_embedding(self, prompt: str) -> List[float]:
        response = self._get_client().embeddings.create(
            model=self.model,
            input=prompt,
        )
        return response.data[0].embedding

In [12]:
EMBEDDING_MODEL="embedding-3"

zhipu_llm = ChatGLM(model=DEFAULT_MODEL, reuse_client=True, api_key=API_KEY, base_url=BASE_URL)

zhipu_embedding = ChatGLMEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY, base_url=BASE_URL)

print(f"LLM模型: {zhipu_llm.metadata.model_name}")
print(f"Embedding模型: {zhipu_embedding.model}")

LLM模型: GLM-4
Embedding模型: embedding-3


In [13]:
from llama_index.core import SimpleDirectoryReader

docs_path = "../docs/"

reader = SimpleDirectoryReader(docs_path)
docs = reader.load_data()

print(f"读取到 {len(docs)} 个文档")
docs

读取到 1 个文档


[Document(id_='37f1881a-0299-43dc-9464-db788aff4cf5', embedding=None, metadata={'file_path': 'd:\\八一农大\\day03\\..\\docs\\test.txt', 'file_name': 'test.txt', 'file_type': 'text/plain', 'file_size': 275, 'creation_date': '2026-07-01', 'last_modified_date': '2026-07-01'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='张老师有一个学生叫张小娟。小娟是计算机科学专业的学生，今年大三。她的成绩非常优秀，尤其是在人工智能课程上表现突出。小娟喜欢编程，经常参加各种编程比赛。她的梦想是成为一名优秀的AI工程师。', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}')]

In [14]:
from llama_index.core import VectorStoreIndex

vector = VectorStoreIndex.from_documents(docs, embed_model=zhipu_embedding)

print("向量索引构建完成")

向量索引构建完成


In [15]:
query_engine = vector.as_query_engine(llm=zhipu_llm)

print("查询引擎创建完成")

查询引擎创建完成


In [16]:
response = query_engine.query("根据上下文回答问题，简明扼要。小娟姓什么？")
print(response.response)

张
